# 🕸️ 02 — Network Analysis of the Crypto Market

The coin market seen as a **network**: who is connected to whom, which coins sit at the centre, which groups move
together, how the structure changes in crises, and how shocks travel. All on the **hourly** Binance data (30 model
coins, notebook 01 Part II), with every network built from **trailing windows only**.

| Network | Built from | Direction | Used for |
|---|---|---|---|
| **Correlation** | 168-hour correlations of deseasonalised returns | undirected | eigenvalues, communities, MST, topology |
| **Spillover (Diebold–Yilmaz)** | generalized FEVD of a VAR (30-day windows; the test window for rankings) | directed | spillover index, NET ranking, PageRank |
| **Shock transmission** | P(coin *j* shocks within 1 hour after coin *i* shocks), 90-day windows | directed, time-ordered | PageRank, contagion baseline |

1. **The market mode** — why one factor dominates.
2. **Spillover connectedness** (Diebold–Yilmaz) — hourly, plus the daily cross-check.
3. **Centrality** — rankings and how they move over time.
4. **Communities** vs sectors.
5. **Minimum spanning tree** and tree length.
6. **Topology around crises.**
7. **Sectors and contagion.**
8. **Case study: the FTX collapse.**
9. **Independent cascade baseline** — the contagion benchmark the TGN must beat.
10. **Network features for the models.**

Outputs come from `cryptograph-data --config configs/data/hourly.yaml --stage network`
(`data/processed/v2/1h/network_features.pt`, `data/processed/v2/rankings.csv`, `reports/network_timeseries_1h.csv`,
`reports/cascade_baseline.csv`); this notebook reads them and adds the figures.

In [1]:
import sys, json
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.metrics import normalized_mutual_info_score

from cryptograph.data.features.targets import future_any
from cryptograph.network.cascade import transmission_counts
from cryptograph.network.mst import mst_edges

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"

V2 = "../data/processed/v2"
hp = torch.load(f"{V2}/1h/panel.pt", weights_only=False)
nf = torch.load(f"{V2}/1h/network_features.pt", weights_only=False)
ts = pd.read_csv("../reports/network_timeseries_1h.csv", parse_dates=["time"]).set_index("time")
rk = pd.read_csv(f"{V2}/rankings.csv")
rk_meta = json.loads(Path(f"{V2}/rankings.json").read_text())
cas = pd.read_csv("../reports/cascade_baseline.csv")
sectors = pd.read_csv("../data/external/sectors.csv").set_index("coin")["sector"]
events = pd.read_csv("../data/external/events.csv", parse_dates=["date"])
events = events[events["date"] >= "2020-01-01"]

times = pd.to_datetime(hp["times"])
split = hp["split_id"].numpy()
SPLITS = ["train", "val", "test"]
gcoins, gidx = nf["graph_coins"], nf["graph_index"]
r = pd.DataFrame(hp["r_ds"].numpy()[:, gidx], index=times, columns=gcoins)    # deseasonalised hourly returns
sec = np.array([sectors[c] for c in gcoins])
SECTOR_COLORS = dict(zip(sorted(set(sec)), px.colors.qualitative.Set2 + px.colors.qualitative.Pastel1))
w0, w1 = rk_meta["window"]

def add_events(fig, size=9, **kw):
    for _, e in events.iterrows():
        fig.add_vline(x=e["date"], line_width=1, line_dash="dash", line_color="#ff5252", opacity=0.6, **kw)
        fig.add_annotation(x=e["date"], y=1, yref="paper", text=e["name"].split("/")[0].strip()[:26], showarrow=False,
                           textangle=-90, xanchor="right", yanchor="top", font=dict(color="#ff5252", size=size))
    return fig

print(f"📊 {len(gcoins)} coins, {len(nf['snapshot_t'])} daily network snapshots ({nf['times'][0][:10]} → {nf['times'][-1][:10]}); "
      f"ranking window {w0[:10]} → {w1[:10]} (test period, all 30 coins listed)")
print("🏷️ sectors:", pd.Series(sec).value_counts().to_dict())

📊 30 coins, 2456 daily network snapshots (2020-01-07 → 2026-09-27); ranking window 2024-04-02 → 2026-09-27 (test period, all 30 coins listed)
🏷️ sectors: {'smart-contract platform': 11, 'payments': 5, 'DeFi / oracle': 5, 'meme': 3, 'layer 2': 2, 'exchange token': 1, 'privacy': 1, 'infrastructure / interoperability': 1, 'identity / AI': 1}


*What this shows:* Loads the network outputs: 2,456 daily network snapshots of the 30 model coins (2020-01-07 → 2026-09-27), each
built from the trailing hours only. The **rankings** use the test period from 2024-04-02, when the last of the 30 coins
(ENA) was listed, so every coin is ranked on the same hours. The sector labels are hand-curated
(`data/external/sectors.csv`): 11 smart-contract platforms, 5 payment coins, 5 DeFi/oracle, 3 meme coins, 2 layer-2s
and 4 single-coin sectors.

## 1. The Market Mode

Eigenvalues of the correlation matrix (test window). If returns were independent noise, every eigenvalue would lie
below the random-matrix (Marchenko–Pastur) bound; eigenvalues above it are real structure.

In [2]:
Rt = r.loc[w0:w1].dropna()
C = Rt.corr().to_numpy()
eig = np.linalg.eigvalsh(C)[::-1]
mp = (1 + np.sqrt(C.shape[0] / len(Rt))) ** 2
fig = px.bar(x=np.arange(1, 11), y=eig[:10], text=[f"{v:.1f}" for v in eig[:10]], log_y=True,
             labels={"x": "eigenvalue rank", "y": "eigenvalue (log scale)"},
             title=f"Top eigenvalues of the 30-coin correlation matrix (test window, {len(Rt):,} hours)")
fig.add_hline(y=mp, line_dash="dot", annotation_text=f"random-noise bound {mp:.2f}")
fig.update_layout(height=340)
fig.show()
print(f"Largest eigenvalue = {eig[0]:.1f} of {C.shape[0]} → the market mode explains {eig[0] / C.shape[0]:.0%} of all variance; "
      f"{int((eig > mp).sum())} eigenvalues are above the noise bound.")

Largest eigenvalue = 19.3 of 30 → the market mode explains 64% of all variance; 1 eigenvalues are above the noise bound.


*What this shows:* One eigenvalue towers over the rest: 19.3 out of 30, so a **single market factor explains 64% of all hourly
variance** across the 30 coins. It is the only eigenvalue clearly above the random-noise bound (1.08). In practice,
most of what any coin does in an hour is "the market moved". Any group structure is small next to it, which is why the
community analysis (section 4) first removes this market mode.

## 2. Spillover Connectedness (Diebold–Yilmaz)

A VAR on the 30 coins' hourly returns; the **generalized FEVD** (order-invariant) gives θ<sub>ij</sub> = the share of coin
*i*'s 24-hour forecast-error variance caused by shocks to coin *j*. **TO** = what a coin sends to the others, **FROM**
= what it receives, **NET** = TO − FROM, **total spillover index (TSI)** = the average share coming from other coins.

In [3]:
d = rk.sort_values("dy_net", ascending=False)
fig = px.bar(d, x="coin", y="dy_net", color="sector", color_discrete_map=SECTOR_COLORS,
             title=f"NET spillover, hourly, test window (TSI = {rk_meta['total_spillover_index']:.1f}%, VAR lag {rk_meta['var_lag_order_bic']} by BIC)")
fig.update_layout(height=400, xaxis_title=None, yaxis_title="NET (percentage points)", xaxis={"categoryorder": "total descending"})
fig.show()

rho = rk[["dy_net", "eigenvector"]].corr(method="spearman").iloc[0, 1]
fig = px.scatter(rk, x="eigenvector", y="dy_net", text="coin", color="sector", color_discrete_map=SECTOR_COLORS,
                 title=f"NET spillover vs eigenvector centrality of the correlation network (Spearman ρ = {rho:.2f})")
fig.update_traces(textposition="top center", marker_size=9)
fig.update_layout(height=420, showlegend=False)
fig.show()

tsi = ts["total_spillover"].rolling(7, min_periods=3).mean()
fig = px.line(x=tsi.index, y=tsi.values, labels={"x": "", "y": "TSI (%)"},
              title="Total spillover index over time (30-day windows, VAR(1), 7-day average)")
add_events(fig)
fig.update_layout(height=380)
fig.show()
display(ts.groupby("split")["total_spillover"].describe()[["mean", "min", "max"]].loc[SPLITS].round(1))

,mean,min,max
split,,,
train,86.1,66.5,92.9
val,90.2,86.2,93.0
test,91.3,82.0,94.8


*What this shows:* The hourly spillover index is very high (**TSI 91.6%**): over 24 hours, more than nine-tenths of each coin's
forecast uncertainty comes from the other coins. LINK, DOT and AVAX are the largest net transmitters; TRX, ZEC and
QNT the largest net receivers. **But the scatter shows the catch:** the NET ranking is *identical* to eigenvector
centrality of the correlation network (Spearman ρ = 1.00). With hourly data, BIC picks one lag and the lagged effects
are tiny (notebook 01 §15), so the generalized FEVD is driven by same-hour correlations. Hourly NET therefore measures
**how tightly a coin is tied to the market**, not who moves first. TRX, ZEC and QNT are "receivers" only because they
follow their own path. Over time the TSI rises from 86% (train) to 90–91% (val/test) and jumps after crises (it
dropped as low as 66% in a calm 2020 window).

### 2b. Daily cross-check (moved from `03_statistical_spillover_baseline`)

The corrected daily Diebold–Yilmaz analysis: Panel A, the 20 coins with complete daily data from 2017-11-09, VAR on
raw daily log returns, 10-day horizon. At daily frequency there is time for lagged effects, so this is where the
spillover *direction* has more content.

In [4]:
from cryptograph.analysis import VARSpilloverBaseline

panel_d = torch.load("../data/processed/v1/panel.pt", weights_only=False)
close_d = pd.read_parquet("../data/interim/master.parquet")["close"].unstack("coin")
panel_a = [c for c, a in zip(panel_d["coins"], panel_d["panel_A"]) if a]
ret_d = np.log(close_d[panel_a]).diff().loc["2017-11-09":].dropna()
baseline = VARSpilloverBaseline(maxlags=2, forecast_horizon=10).fit(ret_d)
dy_d = baseline.compute_spillover_table()
net_d = dy_d["net_spillover"].sort_values(ascending=False)
fig = px.bar(x=net_d.index, y=net_d.values, color=np.where(net_d.values > 0, "net transmitter", "net receiver"),
             color_discrete_map={"net transmitter": "#00e676", "net receiver": "#ff1744"},
             labels={"x": "", "y": "NET (percentage points)", "color": ""},
             title=f"Daily NET spillover, Panel A ({len(panel_a)} coins, {len(ret_d):,} days; TSI = {dy_d['total_spillover_index']:.1f}%)")
fig.update_layout(height=380)
fig.show()
both = sorted(set(net_d.index) & set(rk["coin"]))
rho_d = pd.Series(net_d[both]).corr(rk.set_index("coin").loc[both, "dy_net"], method="spearman")
print(f"Coins in both: {', '.join(both)} → Spearman ρ between the daily and hourly NET rankings: {rho_d:.2f}")

[2026-10-01 12:37:09] [INFO] [cryptograph.analysis.var_spillover] - Fitted VAR baseline with lag order: 2


C:\Users\HARDIK\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


Coins in both: ADA, BCH, BNB, BTC, DOGE, ETC, ETH, LINK, LTC, TRX, XLM, XRP, ZEC → Spearman ρ between the daily and hourly NET rankings: 0.21


*What this shows:* The corrected daily analysis (formerly notebook 03): TSI 86.0% with a VAR(2) over 3,244 days. Its NET ranking
agrees only weakly with the hourly one on the 13 coins they share (Spearman ρ = 0.21). The two answer different
questions. Daily data leave room for one- and two-day lagged spillovers, while hourly NET is a same-hour
correlation measure. **Consequence for TempME validation:** "who transmits shocks" depends on the time scale, so
explanations will be compared against both rankings, and against the time-ordered shock-transmission ranking
below, rather than treating one of them as the ground truth.

## 3. Centrality

Rankings on the test window from three directed and undirected views, then how centrality moves over time.
**PageRank** is computed on the *directed* networks only, on the "depends on" graph (an edge from *i* to *j* when *i*
is driven by *j*), so rank collects at the coins the others depend on.

In [5]:
cols = {"dy_net": "DY NET", "dy_pagerank": "PageRank (spillover)", "shock_pagerank": "PageRank (shock transmission)",
        "eigenvector": "eigenvector (correlation)", "mst_degree": "MST degree"}
sp = rk[list(cols)].rename(columns=cols).corr(method="spearman")
fig = px.imshow(sp.round(2), text_auto=True, zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
                title="How much the rankings agree (Spearman ρ across the 30 coins, test window)")
fig.update_layout(height=420)
fig.show()
top = rk.assign(**{f"{v}": rk[k].rank(ascending=False, method="min").astype(int) for k, v in cols.items()})
display(top.set_index("coin")[list(cols.values())].sort_values("DY NET").head(10))

fi = {n: i for i, n in enumerate(nf["feature_names"])}
X = nf["X"].numpy()
st_times = pd.to_datetime(nf["times"])
show = ["BTC", "ETH", "SOL", "XRP", "DOGE", "LINK"]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("PageRank on the spillover network (1 = average coin)",
                                    "PageRank on the shock-transmission network (1 = average coin)"))
for k, c in enumerate(show):
    j = gcoins.index(c)
    for row, f in ((1, "dy_pagerank"), (2, "shock_pagerank")):
        y = pd.Series(X[:, j, fi[f]], index=st_times).rolling(30, min_periods=10).mean()
        fig.add_trace(go.Scatter(x=y.index, y=y.values, name=c, legendgroup=c, showlegend=row == 1,
                                 line=dict(color=px.colors.qualitative.Plotly[k])), row=row, col=1)
fig.update_layout(height=560, title="Centrality over time (30-day averages)")
fig.show()

,DY NET,PageRank (spillover),PageRank (shock transmission),eigenvector (correlation),MST degree
coin,,,,,
LINK,1,1,8,1,4
DOT,2,2,12,2,1
AVAX,3,3,4,3,7
ADA,4,5,5,6,4
ETH,5,4,9,4,2
DOGE,6,6,14,5,7
ETC,7,7,1,7,4
ARB,8,8,19,8,3
SHIB,9,9,7,9,7


*What this shows:* The three correlation-based rankings (DY NET, PageRank on the spillover network, eigenvector) are the same
ranking (ρ = 1.00); the MST degree agrees moderately (ρ ≈ 0.8). **PageRank on the shock-transmission network**, the
only one built from time order (coin *i* shocks, then coin *j* within the hour), is related but different (ρ ≈ 0.73).
It puts ETC, XRP, POL, AVAX and ADA on top and ZEC, TRX, ENA, QNT and BCH at the bottom, and BTC only 23rd: BTC's own
shocks are not followed by unusually many others. The differences between coins are small (scaled PageRank roughly
0.6–1.3 around the average of 1), and they drift. SOL's shock-transmission PageRank rose from 0.57 in 2020 to
about 1.0–1.2 from 2022, and DOGE's spillover PageRank from 0.7 to 1.2. That drift is why the models get these as
time-varying features, not as a fixed coin label.

## 4. Communities vs Sectors

Louvain (networkx, seeded) on the correlations **with the market mode removed** — otherwise everything is one
community. Do the communities match the sectors (payments, smart-contract platforms, DeFi, meme…)? **NMI** = normalized
mutual information (0 = unrelated, 1 = identical); the null shuffles the sector labels.

In [6]:
comm = rk.set_index("coin")["community"]
tab = (rk.groupby("community").apply(lambda d: ", ".join(f"{c} ({s})" for c, s in zip(d["coin"], d["sector"])), include_groups=False)
       .rename("coins (sector)").to_frame())
display(tab)
nmi_obs = normalized_mutual_info_score(rk["sector"], rk["community"])
rng = np.random.default_rng(0)
null = [normalized_mutual_info_score(rng.permutation(rk["sector"].to_numpy()), rk["community"]) for _ in range(2000)]
print(f"NMI(communities, sectors) on the test window = {nmi_obs:.2f}; shuffled sectors: mean {np.mean(null):.2f}, "
      f"95th percentile {np.quantile(null, 0.95):.2f} → p = {np.mean(np.array(null) >= nmi_obs):.3f}")

sm = ts[["modularity", "nmi_sector", "n_communities"]].rolling(14, min_periods=5).mean()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Modularity (residual correlations)", "NMI with sectors"))
fig.add_trace(go.Scatter(x=sm.index, y=sm["modularity"], name="modularity"), row=1, col=1)
fig.add_trace(go.Scatter(x=sm.index, y=sm["nmi_sector"], name="NMI"), row=2, col=1)
add_events(fig, row="all", col=1)
fig.update_layout(height=520, showlegend=False, title="Community structure over time (14-day averages)")
fig.show()

,coins (sector)
community,
0,"DOT (smart-contract platform), ETC (smart-cont..."
1,"AVAX (smart-contract platform), SOL (smart-con..."
2,"LINK (DeFi / oracle), ETH (smart-contract plat..."
3,"ADA (smart-contract platform), XRP (payments),..."
4,"DOGE (meme), SHIB (meme), PEPE (meme)"


NMI(communities, sectors) on the test window = 0.47; shuffled sectors: mean 0.32, 95th percentile 0.39 → p = 0.003


*What this shows:* After removing the market mode, Louvain finds **5 communities** on the test window, and some are easy to
read: the **meme coins** (DOGE, SHIB, PEPE) form their own group; **BTC, ETH and the Ethereum DeFi tokens** (LINK, AAVE,
UNI) plus BNB form another; a third holds the **newer high-beta platforms** (SOL, AVAX, NEAR, SUI, plus JUP, WLD, ENA
and, less obviously, ZEC); a fourth the **older "payments" coins** (XRP, XLM, ADA, HBAR). The match with sectors is real
but partial: NMI 0.47 vs 0.32 for shuffled sectors (p = 0.003). Communities reflect *how coins trade* (era, exchange
listings, investor base) as much as *what they do*. Modularity averages 0.41–0.46 and **falls after crises**, when
the groups merge into one market move.

## 5. Minimum Spanning Tree

The MST keeps the 29 strongest links (correlation distance d = √(2(1−ρ))) that connect all 30 coins (Mantegna 1999).
Its **normalized tree length** (mean link distance, Onnela et al. 2003) shrinks when the market moves as one block.

In [7]:
edges = mst_edges(C, np.ones(len(gcoins), bool))
G = nx.Graph()
G.add_weighted_edges_from((gcoins[i], gcoins[j], d) for i, j, d in edges)
pos = nx.kamada_kawai_layout(G, weight="weight")
fig = go.Figure()
for a, b, dd in G.edges(data=True):
    fig.add_trace(go.Scatter(x=[pos[a][0], pos[b][0]], y=[pos[a][1], pos[b][1]], mode="lines",
                             line=dict(width=1 + 4 * (1.2 - dd["weight"]), color="#888"), hoverinfo="skip", showlegend=False))
for s_name, col in SECTOR_COLORS.items():
    nodes = [c for c in G.nodes if sectors[c] == s_name]
    if nodes:
        fig.add_trace(go.Scatter(x=[pos[c][0] for c in nodes], y=[pos[c][1] for c in nodes], mode="markers+text",
                                 text=nodes, textposition="top center", name=s_name,
                                 marker=dict(size=[10 + 4 * G.degree(c) for c in nodes], color=col, line=dict(width=1, color="white"))))
fig.update_layout(height=600, title=f"MST of the 30 coins (test window); node size = number of links, thicker = more correlated",
                  xaxis=dict(visible=False), yaxis=dict(visible=False))
fig.show()
print("Hubs (most MST links):", ", ".join(f"{c} {k}" for c, k in sorted(G.degree, key=lambda x: -x[1])[:5]))

tl = ts["tree_length"].rolling(7, min_periods=3).mean()
fig = px.line(x=tl.index, y=tl.values, labels={"x": "", "y": "normalized tree length"},
              title="MST normalized tree length over time (168-hour windows, 7-day average)")
add_events(fig)
fig.update_layout(height=360)
fig.show()

Hubs (most MST links): DOT 11, ETH 5, ARB 4, LINK 3, ADA 3


*What this shows:* The strongest links are DOGE–SHIB and BTC–ETH. The meme coins form one branch (DOGE–SHIB–PEPE), matching the
community result, and ETH is the hub of the "majors" (BTC, LINK, SOL, AAVE, BNB). **DOT has the most links (11)**, but
mostly to loosely attached coins (TRX, ZEC, QNT, HBAR, ICP; distances 0.76–1.06) whose nearest neighbour happens to be
DOT. It is a meeting point for outliers rather than a market leader. The normalized tree length (average 0.66) **shrinks in every crisis**:
down to 0.43 right after the COVID crash, and to its lowest values (0.36) in April 2024. A short tree means all
coins are close together, so diversification across coins disappears exactly when it is needed.

## 6. Topology Around Crises

How the thresholded correlation network (|ρ| ≥ the fitted threshold, notebook 01 §16) changes over time, and in the
two weeks before vs after each crisis in the hourly period.

In [8]:
mets = {"density": "density", "avg_clustering": "average clustering", "avg_shortest_path": "average shortest path",
        "max_core": "largest k-core"}
sm = ts[list(mets)].rolling(7, min_periods=3).mean()
fig = make_subplots(rows=2, cols=2, shared_xaxes=True, subplot_titles=list(mets.values()), vertical_spacing=0.1)
for k, m in enumerate(mets):
    fig.add_trace(go.Scatter(x=sm.index, y=sm[m], name=m, showlegend=False), row=k // 2 + 1, col=k % 2 + 1)
fig.update_layout(height=520, title="Topology of the hourly correlation network (7-day averages)")
fig.show()

cols_ev = {"mean_corr": "mean corr", "tree_length": "tree length", "density": "density", "modularity": "modularity",
           "total_spillover": "TSI %", "mean_shock_lift": "shock lift"}
rows = []
for _, e in events.iterrows():
    before = ts.loc[e["date"] - pd.Timedelta(days=14): e["date"] - pd.Timedelta(hours=1), list(cols_ev)].mean()
    after = ts.loc[e["date"]: e["date"] + pd.Timedelta(days=14), list(cols_ev)].mean()
    rows.append({"event": f"{e['date'].date()} {e['name'][:30]}", **{f"{v} Δ": after[k] - before[k] for k, v in cols_ev.items()}})
evt = pd.DataFrame(rows).set_index("event")
display(evt.round(3).style.background_gradient(cmap="RdBu_r", axis=0).format("{:+.3f}"))

,mean corr Δ,tree length Δ,density Δ,modularity Δ,TSI % Δ,shock lift Δ
event,,,,,,
2020-03-12 COVID Black Thursday,+0.174,-0.202,+0.441,-0.056,+7.650,+0.039
2021-05-19 China bans financial instituti,+0.237,-0.260,+0.543,-0.024,+4.263,+0.008
2022-05-09 Terra UST depeg / LUNA collaps,+0.075,-0.094,+0.236,-0.013,+1.297,+0.026
2022-06-13 Celsius withdrawal freeze / Th,+0.079,-0.108,+0.332,-0.039,+0.233,+0.015
2022-11-08 FTX collapse,+0.124,-0.106,+0.335,-0.050,+3.452,+0.030
2023-03-10 SVB failure / USDC depeg,+0.032,-0.047,+0.150,-0.015,+1.567,+0.002
2024-01-10 US spot BTC ETF approval,+0.082,-0.073,+0.083,-0.017,+0.929,+0.008
2024-08-05 Yen carry-trade unwind,+0.106,-0.126,+0.342,-0.025,+0.722,+0.025
2025-10-10 Oct-10 liquidation cascade,+0.068,-0.121,+0.123,+0.009,+2.398,+0.018


*What this shows:* Across all 9 crises in the hourly period the pattern is the same. In the two weeks after the event the
**mean correlation rises** (+0.03 to +0.24), **density rises** (+0.08 to +0.54), the **tree shrinks**, the **spillover
index rises** (+0.2 to +7.7 points) and the **chance that a shock is followed by another coin's shock rises**. Modularity
falls in 8 of the 9. The biggest structural shifts were the COVID crash and the May 2021 China ban; the SVB/USDC
episode barely moved the network. Over the whole period the test-period network is denser and more clustered than
train (density 0.35 vs 0.30, clustering 0.57 vs 0.47).

## 7. Sectors and Contagion

Are links more common inside a sector (**assortativity** > 0) — and do shocks spread more within a sector than
across sectors? The contagion rate is P(coin *j* shocks within 1 hour | coin *i* shocked), over the split's sample hours.

In [9]:
display(ts.groupby("split")["attr_assortativity"].agg(["mean", "std"]).loc[SPLITS].round(3)
        .rename(columns={"mean": "sector assortativity (mean)", "std": "std"}))

shock = hp["shock_today"].numpy()[:, gidx]
valid = hp["shock_today_mask"].numpy()[:, gidx]
src = shock & valid
anyf, allv = future_any(src, valid, 1)
same = (sec[:, None] == sec[None, :]) & ~np.eye(len(sec), dtype=bool)
cross = sec[:, None] != sec[None, :]
rows = []
for s in SPLITS:
    n, h, bn, bh = transmission_counts(src, anyf, allv, hp["sample_mask"][s].numpy())
    rows += [{"split": s, "pairs": "same sector", "rate": h[same].sum() / n[same].sum()},
             {"split": s, "pairs": "different sector", "rate": h[cross].sum() / n[cross].sum()},
             {"split": s, "pairs": "base rate (any hour)", "rate": bh.sum() / bn.sum()}]
cs = pd.DataFrame(rows)
fig = px.bar(cs, x="split", y="rate", color="pairs", barmode="group", text=[f"{v:.1%}" for v in cs["rate"]],
             title="Chance that coin j shocks within 1 hour after coin i shocks")
fig.update_layout(height=360, yaxis_tickformat=".0%")
fig.show()

n, h, bn, bh = transmission_counts(src, anyf, allv, hp["sample_mask"]["test"].numpy())
by = []
for s_name in sorted(set(sec)):
    m = sec == s_name
    if m.sum() >= 2:
        within = (m[:, None] & m[None, :]) & ~np.eye(len(sec), dtype=bool)
        out_ = m[:, None] & ~m[None, :]
        by.append({"sector": s_name, "coins": int(m.sum()), "within-sector rate": h[within].sum() / n[within].sum(),
                   "to other sectors": h[out_].sum() / n[out_].sum()})
display(pd.DataFrame(by).set_index("sector").style.format({"within-sector rate": "{:.1%}", "to other sectors": "{:.1%}"}))

,sector assortativity (mean),std
split,,
train,-0.137,0.194
val,-0.067,0.115
test,-0.050,0.098


,coins,within-sector rate,to other sectors
sector,,,
DeFi / oracle,5,10.2%,10.4%
layer 2,2,11.5%,11.0%
meme,3,13.7%,10.9%
payments,5,10.9%,10.4%
smart-contract platform,11,10.8%,10.7%


*What this shows:* Sectors matter less than expected. **Sector assortativity is slightly negative** (−0.14 on train, −0.05 on
test): strong links are not more common within a sector. More importantly for contagion, **after a shock the chance
that another coin shocks within an hour is the same inside and across sectors** (train 9.3% vs 9.3%, test 10.8% vs
10.4%, against a 4% base rate). The one exception is **meme coins**, which pass shocks to each other more (13.7%) than to
other sectors (10.9%). So shocks spread market-wide. The sector id stays as a node feature, but the graph should not
be built around sectors.

## 8. Case Study: the FTX Collapse (November 2022)

FTT (FTX's exchange token) is an **event coin**: it is not a model input, but its collapse is one of the crises the
models must handle. Hour by hour: when FTT crashed, did the other coins follow later, or move in the same hour?

In [10]:
c_all = hp["coins"]
z_all = pd.DataFrame(hp["shock_z"].numpy(), index=times, columns=c_all)
thr = hp["shock_threshold"]
win = slice("2022-11-06", "2022-11-11")
close_h = pd.read_parquet("../data/interim/hourly/master_1h.parquet")["close"].unstack("coin").reindex(columns=c_all)
px_w = close_h.loc[win, ["FTT", "BNB", "SOL", "BTC", "ETH"]]
fig = px.line(100 * px_w / px_w.iloc[0], labels={"value": "price (start = 100)", "time": "", "coin": ""},
              title="FTT and the market, 6–11 November 2022 (hourly)")
fig.update_layout(height=360)
fig.show()

zw = z_all.loc[win]
model = hp["model_coins"]
breadth = pd.DataFrame({"FTT z": zw["FTT"], "coins shocked down": (zw[model] < -thr).sum(axis=1),
                        "coins shocked up": (zw[model] > thr).sum(axis=1)})
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, row_heights=[0.4, 0.6],
                    subplot_titles=("FTT shock size (z)", "How many of the 30 model coins shocked in that hour"))
fig.add_trace(go.Bar(x=breadth.index, y=breadth["FTT z"], marker_color=np.where(breadth["FTT z"] < 0, "#ff5252", "#00e676"),
                     name="FTT z"), row=1, col=1)
for h_ in (thr, -thr):
    fig.add_hline(y=h_, line_dash="dot", line_color="grey", row=1, col=1)
fig.add_trace(go.Bar(x=breadth.index, y=-breadth["coins shocked down"], marker_color="#ff5252", name="down"), row=2, col=1)
fig.add_trace(go.Bar(x=breadth.index, y=breadth["coins shocked up"], marker_color="#00e676", name="up"), row=2, col=1)
fig.update_layout(height=520, barmode="relative", showlegend=False, title="FTX collapse hour by hour")
fig.show()

mi = [c_all.index(c) for c in model]
st_tr, sv_tr = hp["shock_today"].numpy()[split == 0][:, mi], hp["shock_today_mask"].numpy()[split == 0][:, mi]
expected = len(model) * (st_tr & sv_tr).sum() / sv_tr.sum()                 # train shock rate per coin-hour
key = breadth[(breadth["FTT z"].abs() > thr) | (breadth[["coins shocked down", "coins shocked up"]].max(axis=1) >= 10)]
display(key.round(1))
big_ftt = breadth[breadth["FTT z"].abs() > 2 * thr]
print(f"Chance level: about {expected:.1f} of 30 coins shock in a random hour. FTT's biggest hours (|z| > {2 * thr:.1f}) "
      f"and the coins shocked in that same hour → next hour:")
for t_, row in big_ftt.iterrows():
    nxt = breadth.loc[t_ + pd.Timedelta(hours=1)] if t_ + pd.Timedelta(hours=1) in breadth.index else None
    n_next = int(nxt["coins shocked down"] + nxt["coins shocked up"]) if nxt is not None else -1
    print(f"  {t_:%d %b %H:00}  FTT z {row['FTT z']:+.1f}: {int(row['coins shocked down'] + row['coins shocked up'])} same hour → {n_next} next hour")

,FTT z,coins shocked down,coins shocked up
2022-11-06 01:00:00,-3.0,1,0
2022-11-06 05:00:00,-3.8,0,0
2022-11-06 08:00:00,2.3,0,0
2022-11-06 12:00:00,2.7,0,0
2022-11-06 14:00:00,6.0,0,0
2022-11-06 15:00:00,-14.1,0,0
2022-11-06 16:00:00,4.1,0,0
2022-11-06 23:00:00,-0.8,18,0
2022-11-07 07:00:00,0.5,17,0
2022-11-07 15:00:00,2.4,0,2


Chance level: about 1.2 of 30 coins shock in a random hour. FTT's biggest hours (|z| > 4.6) and the coins shocked in that same hour → next hour:
  06 Nov 14:00  FTT z +6.0: 0 same hour → 0 next hour
  06 Nov 15:00  FTT z -14.1: 0 same hour → 0 next hour
  08 Nov 02:00  FTT z -10.4: 18 same hour → 18 next hour
  08 Nov 04:00  FTT z -6.3: 11 same hour → 1 next hour
  08 Nov 06:00  FTT z +5.4: 5 same hour → 0 next hour
  08 Nov 16:00  FTT z +8.1: 20 same hour → 24 next hour
  08 Nov 17:00  FTT z -17.0: 24 same hour → 24 next hour
  08 Nov 18:00  FTT z -13.1: 24 same hour → 14 next hour
  08 Nov 19:00  FTT z -7.5: 14 same hour → 0 next hour


*What this shows:* FTT's own collapse began **without the market**: on 6 November at 15:00 UTC (the hour Binance's CEO announced
it would sell its FTT) FTT fell 14σ and **no** model coin shocked in that hour or the next. The market reacted on
**8 November**. At 02:00 FTT fell 10σ and 18 of the 30 coins shocked in the *same* hour. At 16:00, around Binance's
announced takeover, 20 coins jumped together. At 17:00–18:00 the deal unravelled: FTT fell 17σ and all 24
coins trading then shocked down. In a random hour only about 1.2 of the 30 coins shock. There were also market-wide
shock hours without an FTT move (6 Nov 23:00, 7 Nov 07:00, 9 Nov). Contagion here was **simultaneous and news-driven**:
it arrived when the crisis became a market-wide story, not as a chain from FTT to one coin to the next. This matches
the missing hourly lead-lag (notebook 01 §15).

## 9. Independent Cascade Baseline

The **independent cascade** model (Kempe, Kleinberg & Tardos 2003): each shocked coin *i* gets one chance to
"activate" coin *j* with probability p<sub>ij</sub>, estimated on **train only** (shrunk towards *j*'s base rate). With
several coins shocked in the same hour, P(*j* shocks) = 1 − Π(1 − p<sub>ij</sub>). It is scored on exactly the TGN's
evaluation events (each shock → every other coin, label: does *j* shock within *k* hours), against two simpler rules:
the receiver's base rate, and p<sub>ij</sub> alone. This is the bar the TGN has to clear.

In [11]:
c1 = cas[(cas["resolution"] == "1h") & (cas["split"] != "train")]
fig = px.line(c1, x="k_hours", y="auprc", color="model", facet_col="split", markers=True, log_x=True,
              category_orders={"split": ["val", "test"]},
              title="Contagion prediction on the TGN events (1h): AUPRC by horizon k")
for s_i, s in enumerate(["val", "test"]):
    pr = c1[(c1["split"] == s) & (c1["model"] == "receiver base rate")]
    fig.add_trace(go.Scatter(x=pr["k_hours"], y=pr["positive_rate"], mode="lines", line=dict(dash="dot", color="grey"),
                             name="positive rate (random)", showlegend=s_i == 0), row=1, col=s_i + 1)
fig.update_xaxes(tickvals=[1, 3, 6, 12, 24], title="k (hours)")
fig.update_layout(height=400)
fig.show()
tab = c1.pivot_table(index=["split", "k_hours"], columns="model", values=["auprc", "auroc"]).round(3)
display(tab.loc[["val", "test"]])

auprc                                   \
model         independent cascade pairwise p_ij receiver base rate   
split k_hours                                                        
val   1                     0.117         0.089              0.086   
      3                     0.212         0.180              0.174   
      6                     0.307         0.276              0.273   
      12                    0.411         0.404              0.403   
      24                    0.578         0.591              0.592   
test  1                     0.160         0.109              0.107   
      3                     0.280         0.211              0.207   
      6                     0.382         0.316              0.311   
      12                    0.507         0.452              0.450   
      24                    0.672         0.631              0.631   

                            auroc                                   
model         independent cascade pairwise p_ij receiver base rate  
split k_hours                                                       
val   1                     0.605         0.512              0.506  
      3                     0.578         0.516              0.507  
      6                     0.558         0.514              0.509  
      12                    0.526         0.514              0.509  
      24                    0.507         0.518              0.517  
test  1                     0.647         0.512              0.501  
      3                     0.619         0.508              0.502  
      6                     0.600         0.505              0.501  
      12                    0.578         0.505              0.503  
      24                    0.566         0.504              0.503

*What this shows:* The independent cascade clearly beats the base rate at short horizons. At *k* = 1 hour on test its AUPRC is
**0.160 vs 0.107** (AUROC 0.65 vs 0.50), and on validation 0.117 vs 0.086. The edge shrinks with the horizon and is gone
on validation by 24 hours (0.578 vs 0.592). Notice that **p<sub>ij</sub> alone is no better than the base rate**:
knowing *which* coin shocked adds almost nothing. The cascade's gain comes from *how many* coins shock in the same
hour, which fits the market-wide contagion seen above. **This sets the bar for the TGN:** to justify a temporal graph
model it must beat AUPRC 0.16 (test, 1h) and 0.12 (val), and it should do so by learning *who* is at risk, not just
counting simultaneous shocks.

## 10. Network Features for the Models

`network_features.pt` holds, for every daily snapshot and coin, the network position computed from trailing data only
(a model at hour *t* uses the latest snapshot at or before *t*; a unit test checks that changing data after *t* leaves
these features unchanged). Eigenvector centrality and PageRank are rescaled so the average coin = 1.

In [12]:
desc = {"strength": "mean |correlation| with the other coins (168h)",
        "eigenvector": "eigenvector centrality of the correlation network (1 = average)",
        "mst_degree": "number of MST links",
        "community_size_frac": "share of coins in the coin's Louvain community",
        "dy_pagerank": "PageRank on the 30-day spillover network (1 = average)",
        "dy_net": "Diebold–Yilmaz NET spillover, 30-day window (pp)",
        "shock_pagerank": "PageRank on the 90-day shock-transmission network (1 = average)",
        "shock_out_lift": "mean excess chance that another coin shocks within 1h after this coin does"}
snap_split = split[nf["snapshot_t"].numpy()]
act_s = hp["active"].numpy()[nf["snapshot_t"].numpy()][:, gidx]
rows = []
for f, i in fi.items():
    row = {"feature": f, "meaning": desc[f]}
    for k, s in enumerate(SPLITS):
        m = snap_split == k
        row[f"coverage {s}"] = f"{np.isfinite(X[m][:, :, i])[act_s[m]].mean():.0%}"
    rows.append(row)
display(pd.DataFrame(rows).set_index("feature"))
tr = pd.DataFrame(X[snap_split == 0].reshape(-1, X.shape[2]), columns=nf["feature_names"])
fig = px.imshow(tr.corr(method="spearman").round(2), text_auto=True, zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
                title="Network features: Spearman correlation on train (all below the 0.9 redundancy limit)")
fig.update_layout(height=480)
fig.show()
print(f"Tensor X {tuple(X.shape)} (snapshots, coins, features); community ids {tuple(nf['community'].shape)}; "
      f"sector ids for {len(nf['sector_id'])} coins ({len(nf['sector_names'])} sectors)")

,meaning,coverage train,coverage val,coverage test
feature,,,,
strength,mean |correlation| with the other coins (168h),100%,100%,100%
eigenvector,eigenvector centrality of the correlation netw...,100%,100%,100%
mst_degree,number of MST links,100%,100%,100%
community_size_frac,share of coins in the coin's Louvain community,100%,100%,100%
dy_pagerank,PageRank on the 30-day spillover network (1 = ...,97%,99%,100%
dy_net,"Diebold–Yilmaz NET spillover, 30-day window (pp)",97%,99%,100%
shock_pagerank,PageRank on the 90-day shock-transmission netw...,95%,96%,99%
shock_out_lift,mean excess chance that another coin shocks wi...,95%,96%,99%


Tensor X (2456, 30, 8) (snapshots, coins, features); community ids (2456, 30); sector ids for 30 coins (9 sectors)


*What this shows:* Eight node features per snapshot and coin. Coverage of active coins is 100% for the correlation-based features
and 95–100% for the spillover and shock-transmission ones, which need 30 or 90 days of history and at least 5
shocks. No pair is redundant on train (the highest |ρ| is 0.78, eigenvector vs spillover PageRank), so all eight are
kept. Together with the community id and the sector id, they are ready to be joined to the TGN nodes and the
snapshot GNN.

## Key Takeaways

- **One market factor dominates:** it explains 64% of hourly variance; group structure is real but secondary.
- **Hourly Diebold–Yilmaz is a correlation measure:** TSI 91.6%, but NET = eigenvector centrality (ρ = 1.00). The direction of
  spillovers depends on the time scale (daily vs hourly NET ρ = 0.21), so TempME is validated against several rankings,
  including the time-ordered shock-transmission ranking.
- **Communities partly follow sectors** (meme coins, BTC/ETH + DeFi, new platforms, old payment coins; NMI 0.47, p = 0.003),
  but **contagion ignores sectors**: within- and cross-sector rates are the same, except among meme coins.
- **Crises compress the network:** in all 9 crises correlation, density, spillover and shock lift rise and the MST shrinks.
- **Contagion is simultaneous:** in the FTX collapse the market moved in the same hour as FTT's biggest falls, not in a chain.
- **Baseline to beat:** the independent cascade reaches AUPRC 0.16 (test) / 0.12 (val) at 1 hour, against 0.11 / 0.09 for the base rate.
- **Model inputs:** 8 leakage-free network node features per daily snapshot (`network_features.pt`), plus community and sector ids;
  rankings in `data/processed/v2/rankings.csv` for explanation validation only.

**Next:** model plan and the skeleton notebooks 03–06 (TGN vs baselines, TempME explanations, persistence by shock type, robustness).